# Chapter 5 &mdash; State Names as Residues: MSB-First "Divisible by 3"

**Concept 7 of the Chapter 5 decomposition:** *State Names as Residues: MSB-First "Divisible by 3"*

Track $N \bmod 3$, not $N$; the recurrence $N \mapsto 2N+b$ drives every transition.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Residue-States-MSB/Concept-Residue-States-MSB.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Read a binary numeral **most-significant bit first**. Appending bit $b$ maps the value
$N$ to $2N+b$.

You cannot store $N$ &mdash; it is unbounded. You can store $N \bmod 3$, because

$$(2N+b) \bmod 3 = (2(N \bmod 3) + b) \bmod 3.$$

Three states, one per residue, and the transition table is just that formula
evaluated six times. The state name **is** the residue.

## 2. Definitions

### The recurrence, as a function

In [ ]:
def step_resid(r, b, m=3): return (2*r + int(b)) % m
print("residue table (state, bit) -> state:")
for r in range(3):
    for b in '01':
        print("  (%d, %s) -> %d" % (r, b, step_resid(r, b)))

### The DFA, read straight off that table

In [ ]:
Div3 = md2mc('''DFA
IF : 0 -> IF     !! (2*0+0)%3 = 0
IF : 1 -> S1     !! (2*0+1)%3 = 1
S1 : 0 -> S2     !! (2*1+0)%3 = 2
S1 : 1 -> IF     !! (2*1+1)%3 = 0
S2 : 0 -> S1     !! (2*2+0)%3 = 1
S2 : 1 -> S2     !! (2*2+1)%3 = 2
''')

### The reference specification

In [ ]:
def in_Div3(s): return s != '' and int(s, 2) % 3 == 0

<!-- nav-strip -->

---

&larr;&nbsp;[Ch5&nbsp;6.&nbsp;State Names as Compressed History: "Ends with 0101"](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Compressed-History-Suffix/Concept-Compressed-History-Suffix.ipynb) &nbsp;&middot;&nbsp; [**Chapter 5** index](https://github.com/ganeshutah/Jove/blob/master/Chapter5/README.md) &nbsp;&middot;&nbsp; [Ch5&nbsp;8.&nbsp;The Mod Algebra That Makes Residue States Computable](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Mod-Algebra/Concept-Mod-Algebra.ipynb)&nbsp;&rarr;

---

## 3. Tests

The state name is literally $N \bmod 3$.

In [ ]:
tag = {'IF': 0, 'S1': 1, 'S2': 2}
for s in ['', '0', '1', '11', '110', '1001', '10101']:
    v = int(s, 2) if s else 0
    print("%-8r value %-4d N%%3 = %d   state %s" % (s, v, v % 3, run_dfa(Div3, s)))
assert all(tag[run_dfa(Div3, s)] == (int(s,2) if s else 0) % 3
           for s in ['', '0', '1', '11', '110', '1001', '10101', '111111'])

So acceptance is divisibility &mdash; on every numeral up to 12 bits.

In [ ]:
from itertools import product
bad = [''.join(p) for k in range(1, 13) for p in product('01', repeat=k)
       if accepts_dfa(Div3, ''.join(p)) != (int(''.join(p), 2) % 3 == 0)]
print("mismatches on all 1..12-bit numerals :", bad)
assert not bad

Three states handle numerals of any size, including ones Python prints in scientific ranges.

In [ ]:
big = bin(3 * 7**40)[2:]
print("numeral of %d bits, divisible by 3? %s" % (len(big), accepts_dfa(Div3, big)))
assert accepts_dfa(Div3, big)
print("|Q| = %d, regardless." % len(Div3["Q"]))

## 4. Animation

Three residues, six edges &mdash; the whole of modular arithmetic in a picture.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(Div3, FuseEdges=True)

## 5. Exercises


1. Build the divisible-by-5 machine the same way. How many states?
2. Why does reading MSB-first make the recurrence so simple?
3. Which state would be final for "leaves remainder 1"?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter5/Concept-Residue-States-MSB')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')